# NoFluffJobs API search

Query the NoFluffJobs posting search endpoint, inspect the response, and collect the returned postings into a clean list.

## 1. Imports

We use `httpx` for the HTTP request and `pandas` later for tabular display.

In [41]:
import httpx
import pandas as pd

## 2. Endpoint and query parameters

`params` are sent as query string arguments. Salary is requested in EUR per month. The `region`/`requirement` filters are left commented so the search stays broad.

In [42]:
url = "https://nofluffjobs.com/api/search/posting"

# Query parameters sent on the URL
params = {
    "salaryCurrency": "EUR",
    "salaryPeriod": "month",
    # "region": "nl",          # 'nl', 'pl', 'cz', 'hu', etc.
    "language": "en",
    "page": 1,
}

## 3. Request body

The backend expects the key `criteriaSearch` (not `criteria`). Add filters such as `requirement` inside it to narrow the search.

In [43]:
payload = {
    "criteriaSearch": {
        # "requirement": ["python"]
    }
}

## 4. HTTP headers

A browser-like `User-Agent` plus `Referer`/`Origin` are required for the API to accept the request.

In [44]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://nofluffjobs.com/",
    "Origin": "https://nofluffjobs.com",
}

## 5. Send the request

In [45]:
response = httpx.post(url, params=params, json=payload, headers=headers)
print(f"HTTP status: {response.status_code}")

HTTP status: 200


## 6. Parse and clean the response

On success, extract the `postings` list. Each posting is kept as-is for now; `salary` is read out to confirm it is present.

In [46]:
cleaned_postings = []

if response.status_code == 200:
    data = response.json()
    postings = data.get("postings", [])
    print(f"Success! Retrieved {len(postings)} postings.")

    for job in postings:
        salary = job.get("salary", {})
        cleaned_postings.append(job)
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Success! Retrieved 249 postings.


## 7. Inspect the collected postings

In [47]:
display(cleaned_postings)

[{'id': 'power-bi-power-platform-consultant-business-travels-link-group-Remote',
  'name': 'Link Group',
  'location': {'places': [{'city': 'Remote',
     'url': 'power-bi-power-platform-consultant-business-travels-link-group-remote'},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lower-silesian',
     'url': 'power-bi-power-platform-consultant-business-travels-link-group-lower-silesian',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'kuyavian-pomeranian',
     'url': 'power-bi-power-platform-consultant-business-travels-link-group-kuyavian-pomeranian',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lodz',
     'url': 'power-bi-power-platform-consultant-business-travels-link-group-lodz',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lublin',
     'url': 'power-bi-power-platform-consultant-business-travels-link-group

### Optional: view as a DataFrame

Flatten a few key fields for easier inspection.

In [48]:
if cleaned_postings:
    df = pd.DataFrame([
        {
            "title": j.get("title"),
            "company": j.get("name"),
            "salary_from": j.get("salary", {}).get("from"),
            "salary_to": j.get("salary", {}).get("to"),
            "currency": j.get("salary", {}).get("currency"),
            "remote": j.get("fullyRemote"),
        }
        for j in cleaned_postings
    ])
    display(df)
else:
    print("No postings collected.")

,title,company,salary_from,salary_to,currency,remote
0,Power BI / Power Platform Consultant (Business...,Link Group,3360.0,3360.0,EUR,False
1,Power BI / Power Platform Consultant (Business...,Link Group,3360.0,3360.0,EUR,False
2,Power BI / Power Platform Consultant (Business...,Link Group,3360.0,3360.0,EUR,False
3,Power BI / Power Platform Consultant (Business...,Link Group,3360.0,3360.0,EUR,False
4,Power BI / Power Platform Consultant (Business...,Link Group,3360.0,3360.0,EUR,False
...,...,...,...,...,...,...
244,Lider Testów Wydajnościowych,Upvanta,4746.0,5481.0,EUR,False
245,Lider Testów Wydajnościowych,Upvanta,4746.0,5481.0,EUR,False
246,Lider Testów Wydajnościowych,Upvanta,4746.0,5481.0,EUR,False
247,Lider Testów Wydajnościowych,Upvanta,4746.0,5481.0,EUR,False


## 8. Persist to SQLite

Store each posting in a local SQLite database.

In [49]:
import sqlite3
import json as _json

db_path = "job_postings.db"
conn = sqlite3.connect(db_path)

# Flatten every field of each posting into columns, keeping all of them
df_all = pd.json_normalize(cleaned_postings)
if "id" in df_all.columns:
    df_all = df_all.set_index("id")

# Make nested values (lists/dicts) storable in SQLite
df_all = df_all.map(lambda x: _json.dumps(x) if isinstance(x, (list, dict)) else x)

df_all.to_sql("nofluff", conn, if_exists="replace", index=True, index_label="id")
conn.commit()

print(f"Saved {len(df_all)} postings to nofluff table in {db_path}")

cur = conn.cursor()
print(cur.execute("SELECT COUNT(*) FROM nofluff").fetchone())
print([r[1] for r in cur.execute("PRAGMA table_info(nofluff)").fetchall()][:10])
conn.close()


Saved 249 postings to nofluff table in job_postings.db
(249,)
['id', 'name', 'posted', 'renewed', 'title', 'category', 'seniority', 'url', 'regions', 'fullyRemote']
